In [ ]:
print("all ok")  # need to install ipykernel package

all ok


In [2]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [18]:
# if load_dotenv() fails to load the env vars, then this set of code will fix that

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing in your .env file")
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [13]:
from langchain_openai import ChatOpenAI

In [14]:
chat_llm = ChatOpenAI(model="gpt-4o-mini")

In [ ]:
chat_llm_op = chat_llm.invoke(input="Hello, how are you>?")
chat_llm_op

In [15]:
import re

def clean_response(text):
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

### Configuring a LLM using ChatGroq and qwen3-32b model

In [1]:
from langchain_groq import ChatGroq

In [12]:
chat_llm = ChatGroq(
    model="qwen/qwen3-32b",
    groq_api_key=os.getenv("GROQ_API_KEY")
)

In [10]:
chat_llm_output = chat_llm.invoke(input="Hi, How are you?")
chat_llm_output

AIMessage(content='<think>\nOkay, the user asked "Hi, How are you?" I should respond with a friendly and cheerful greeting. I need to keep it natural and avoid being too formal. I\'ll thank them for their concern and mention I\'m doing well. I\'ll add an emoji to make it more lively and end by asking how they\'re doing. That should cover the greeting, expression of gratitude, and show interest in their well-being.\n</think>\n\nHi there! I\'m doing well, thanks for asking! 😊 How about you? I hope your day is off to a great start!', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 14, 'total_tokens': 133, 'completion_time': 0.279060945, 'completion_tokens_details': None, 'prompt_time': 0.000403463, 'prompt_tokens_details': None, 'queue_time': 0.046784047, 'total_time': 0.279464408}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_5cf921caa2', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provide

In [14]:
chat_llm_output.content

'<think>\nOkay, the user asked "Hi, How are you?" I should respond with a friendly and cheerful greeting. I need to keep it natural and avoid being too formal. I\'ll thank them for their concern and mention I\'m doing well. I\'ll add an emoji to make it more lively and end by asking how they\'re doing. That should cover the greeting, expression of gratitude, and show interest in their well-being.\n</think>\n\nHi there! I\'m doing well, thanks for asking! 😊 How about you? I hope your day is off to a great start!'

In [16]:
res = clean_response(chat_llm_output.content)
res

"Hi there! I'm doing well, thanks for asking! 😊 How about you? I hope your day is off to a great start!"

### Defining the State

In [19]:
from typing_extensions import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage

In [ ]:
# state
class GraphState(TypedDict):
    # message is acting as a `key`, and the list as the `value` 
    message: Annotated[list[AnyMessage], operator.add]  # this will be a list of any messages and we can append as many as messages over here.

# using this state only, we are passing the input to diff diff nodes.


### Defining the Nodes

In [ ]:
def llm_call(state: GraphState) -> dict:
    """ Call the LLM using conversation messages and append AI response. """
    response = clean_response(chat_llm.invoke(state["messages"]))  # AIMessage

    return {
        "message": [response]
    }